# ML Zoomcamp 2025 Capstone – Anime Hit Prediction EDA & Model Prep Notebook

1) Problem framing & leakage checks  
2) Data prep + EDA  
3) Baseline ML models (linear + tree)  
4) Deep learning model (PyTorch) **and** a Keras alternative  
5) Export artifacts + **ONNX** (for lightweight inference)  
6) Web service (FastAPI)  
7) Docker  
8) **Serverless** deployment (AWS Lambda container using ONNX Runtime)  
9) **Kubernetes** deployment (kind/minikube)

> Dataset assumption: `details.csv` + `stats.csv` joined on `mal_id` (anime metadata + engagement stats).

## Link to download data and drop in the '/data' folder within the repo


Open a Terminal to download full dataset, extract zip file, and then place the details.csv and stat.csv in the `/data` folder within the repo. The data is currently in the repo but if it is not for some reason here you go.
```bash
curl -L -o ~/Downloads/anime-dataset-jan-1917-to-oct-2025.zip\
  https://www.kaggle.com/api/v1/datasets/download/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025
````

---

In [ ]:

# --- Project root + path resolution (prevents creating artifacts under notebooks/) ---
from pathlib import Path
import os

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        # Heuristic: repo root contains src/ and (README or Makefile)
        if (p / "src").exists() and ((p / "README.md").exists() or (p / "Makefile").exists()):
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data"
ART_DIR = REPO_ROOT / "artifacts"
ART_DIR_SKLEARN = REPO_ROOT / "artifacts_sklearn"
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"

ART_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR_SKLEARN.mkdir(parents=True, exist_ok=True)

print("✅ REPO_ROOT:", REPO_ROOT)
print("✅ DATA_DIR :", DATA_DIR)
print("✅ ART_DIR  :", ART_DIR)
print("✅ NOTEBOOKS_DIR:", NOTEBOOKS_DIR)


## Problem framing: “Hit prediction” (cold-start)

**Goal:** Predict whether an anime will be a “hit” **using only pre-release metadata**.

**The Why:** I like anime which is why i chose this project. Also this something that studios can use to determine if a particular anime or manga will be a hit or not from a business perspective. If I made this an app users could see if the upcoming series is likely to be something they would watch or read.

A practical framing:
- **Input**: title/metadata known before release (type, season/year, genres/themes, rating, source, planned episodes, etc.)
- **Output**: probability of “hit”
- **Use case**: licensing / marketing prioritization, editorial recommendations, portfolio planning.

### Define “hit”
  - Hit: `is_hit = 1` if `members` is in the **top 20%** (or top 10%) of all anime in the dataset.

<br>

> Important: `members` is used **only for labeling during training**. At prediction time, you must NOT use engagement fields.

### Leakage rules

From `stats.csv` and related columns, these are **NOT allowed at inference**:
- `members`, `favorites`
- `watching`, `completed`, `on_hold`, `dropped`, `plan_to_watch`, `total`
- any score-vote breakdown (`score_1_votes`, …)

These are treated as “probably not known pre-release” unless justified:
- `rank`, `popularity`, `scored_by`, sometimes `score`

**Allowed** (typical pre-release):
- `type`, `source`, `rating`, `season`, `year`, `episodes`
- `genres`, `themes`, `demographics`, `studios`
- synopsis text (if I want to do a text based model)

## Setup (local Mac + Apple Silicon notes)

Recommended local environment (conda or venv):
- Python 3.11
- pandas, numpy, scikit-learn
- torch (Apple Silicon: `pip install torch torchvision torchaudio` from PyTorch instructions)
- onnx, onnxruntime (or onnxruntime-silicon locally)

For deployment containers / k8s, we will typically use Linux images.  
On an M-series Mac, build with:
- `docker buildx build --platform linux/amd64 ...`

(This matters especially for Lambda and many k8s nodes.)

---
## 0) Imports


In [ ]:
import os, random, json
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any


pd.set_option("display.max_columns", 200)


In [ ]:
# Reproducibility: We will use fixed seeds across python/numpy/sklearn/torch

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("Global SEED set to:", SEED)


---
## 1) Setting Directory Path for Data
If you run this notebook from `notebooks/`, then `./data` would point to `notebooks/data/`.
This cell searches **upward** until it finds a `data/` folder.


In [ ]:
# --- Robustly find the repo root that contains a 'data/' folder ---
p = Path.cwd()
while p != p.parent and not (p / "data").exists():
    p = p.parent

DATA_DIR = p / "data"
DETAILS_PATH = DATA_DIR / "details.csv"
STATS_PATH   = DATA_DIR / "stats.csv"

print("Working dir:", Path.cwd())
print("Resolved DATA_DIR:", DATA_DIR)
print("Details exists?", DETAILS_PATH.exists(), DETAILS_PATH)
print("Stats exists?  ", STATS_PATH.exists(), STATS_PATH)


---
## 2) Data Load + Merge (details.csv + stats.csv)


In [ ]:
df_details = pd.read_csv(DETAILS_PATH)
df_stats   = pd.read_csv(STATS_PATH)

# Merge on mal_id
df = df_details.merge(df_stats, on="mal_id", how="left")

print("Merged shape:", df.shape)
display(df.head(10))


---
## 3) Exploratory Data Analysis (EDA)

This section includes:
- Histograms: `year`, `episodes`, `score`, and `log1p(members)`
- Top genres/themes
- Correlation heatmap
- Trends over time


In [ ]:
# Missingness overview (top 25)
missing = (df.isna().mean().sort_values(ascending=False) * 100).round(2)
missing = missing[missing > 0]
display(missing.head(25).to_frame("missing_%"))


> Note: We have missing values that have to be reconciled in our data. Will address further down before model training.

In [ ]:
def hist_col(data, col, bins=30, title=None):
    if col not in data.columns:
        print(f"Skipping {col} (not found)")
        return
    x = pd.to_numeric(data[col], errors="coerce").dropna()
    if x.empty:
        print(f"Skipping {col} (all missing)")
        return
    plt.figure(figsize=(8,4))
    plt.hist(x, bins=bins)
    plt.title(title or f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()

hist_col(df, "year", bins=40)
hist_col(df, "episodes", bins=40)
hist_col(df, "score", bins=30)

if "members" in df.columns:
    x = pd.to_numeric(df["members"], errors="coerce").dropna()
    plt.figure(figsize=(8,4))
    plt.hist(np.log1p(x), bins=40)
    plt.title("Distribution of log1p(members)")
    plt.xlabel("log1p(members)")
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()


In [ ]:
def parse_multilabel(x):
    """Handles: lists, stringified lists, or comma-separated strings."""
    if pd.isna(x):
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(i).strip() for i in x if str(i).strip()]
    s = str(x).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            val = ast.literal_eval(s)
            if isinstance(val, (list, tuple, set)):
                return [str(i).strip() for i in val if str(i).strip()]
        except Exception:
            pass
    return [p.strip() for p in s.split(",") if p.strip()]

def top_k_counts(data, col, k=20):
    if col not in data.columns:
        print(f"Skipping {col} (not found)")
        return None
    exploded = data[col].apply(parse_multilabel).explode()
    exploded = exploded.dropna()
    exploded = exploded[exploded.astype(str).str.len() > 0]
    return exploded.value_counts().head(k)

def bar_top(series, title):
    if series is None or series.empty:
        print("Nothing to plot:", title)
        return
    plt.figure(figsize=(10,6))
    plt.barh(series.index[::-1], series.values[::-1])
    plt.title(title)
    plt.xlabel("count")
    plt.tight_layout()
    plt.show()

top_genres = top_k_counts(df, "genres", k=20)
display(top_genres)
bar_top(top_genres, "Top 20 Genres")

top_themes = top_k_counts(df, "themes", k=20)
display(top_themes)
bar_top(top_themes, "Top 20 Themes")


In [ ]:
# Correlation heatmap (numeric)
num = df.select_dtypes(include=[np.number]).copy()
keep = [c for c in num.columns if num[c].notna().mean() >= 0.5]
num = num[keep]

if len(num.columns) >= 2:
    corr = num.corr(numeric_only=True)
    plt.figure(figsize=(10,8))
    plt.imshow(corr.values, aspect="auto")
    plt.title("Correlation heatmap (numeric)")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.colorbar()
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for correlation heatmap.")


In [ ]:
# Trends over time (counts, avg score, median members)
if "year" in df.columns:
    tmp = df.dropna(subset=["year"]).copy()
    tmp["year"] = pd.to_numeric(tmp["year"], errors="coerce").dropna().astype(int)

    per_year = tmp.groupby("year").size()
    plt.figure(figsize=(10,4))
    plt.plot(per_year.index, per_year.values)
    plt.title("Anime count by year")
    plt.xlabel("year")
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()

    if "score" in tmp.columns:
        avg_score = pd.to_numeric(tmp["score"], errors="coerce")
        avg_score = tmp.assign(score=avg_score).groupby("year")["score"].mean()
        plt.figure(figsize=(10,4))
        plt.plot(avg_score.index, avg_score.values)
        plt.title("Average score by year")
        plt.xlabel("year")
        plt.ylabel("avg score")
        plt.tight_layout()
        plt.show()

    if "members" in tmp.columns:
        members = pd.to_numeric(tmp["members"], errors="coerce")
        med_members = tmp.assign(members=members).groupby("year")["members"].median()
        plt.figure(figsize=(10,4))
        plt.plot(med_members.index, np.log1p(med_members.values))
        plt.title("Median members by year (log1p)")
        plt.xlabel("year")
        plt.ylabel("log1p(median members)")
        plt.tight_layout()
        plt.show()
else:
    print("Skipping trends: 'year' not found")


---
## 4) Handling Missing Values

**Why:** Some rows are missing `year` and/or `season`. To reduce missingness (and avoid dropping data), we:
- parse `start_date` as datetime
- fill missing `year` from `start_date.dt.year`
- fill missing `season` from `start_date` month → Winter/Spring/Summer/Fall

If `start_date` is missing, we leave values missing and handle them later with **imputation** or **row dropping**.


In [ ]:
# Parse dates safely
if "start_date" in df.columns:
    df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")

# Fill year from start_date if missing
if "year" in df.columns and "start_date" in df.columns:
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["year"] = df["year"].fillna(df["start_date"].dt.year)

# Derive season from start_date month if season missing
def month_to_season(m):
    if pd.isna(m):
        return np.nan
    m = int(m)
    if m in [12, 1, 2]:
        return "Winter"
    if m in [3, 4, 5]:
        return "Spring"
    if m in [6, 7, 8]:
        return "Summer"
    return "Fall"

if "season" in df.columns and "start_date" in df.columns:
    df["season"] = df["season"].astype("string")
    derived = df["start_date"].dt.month.apply(month_to_season)
    df["season"] = df["season"].fillna(derived)

# Make missing categories explicit (helps models)
for c in ["type", "season", "rating", "source", "status"]:
    if c in df.columns:
        df[c] = df[c].astype("string").fillna("Unknown")

# Episodes often missing; keep numeric but allow later imputer to fill
if "episodes" in df.columns:
    df["episodes"] = pd.to_numeric(df["episodes"], errors="coerce")

for c in ["start_date", "year", "season", "episodes", "type", "rating"]:
    if c in df.columns:
        print(f"{c:>10} missing %:", round(df[c].isna().mean() * 100, 2))


In [ ]:
# Missingness overview (top 25) post data transformation
missing = (df.isna().mean().sort_values(ascending=False) * 100).round(2)
missing = missing[missing > 0]
display(missing.head(25).to_frame("missing_%"))

> Note: Having missing end dates is fine because many anime series don't end (DragonBall Z or One Piece), have not eneded yet, or get put on an indefinite pause.

---
## 5) Clean feature split + Top‑K multi-hot encoding for multi-label fields

We define a **cold-start** target: *hit = top 20% by members*.

**Leakage note:** We use `members` **only** to create the label. We do **not** use engagement columns as features.


In [ ]:
HIT_Q = 0.8  # top 20% by members

df_ml = df.copy()
df_ml["members"] = pd.to_numeric(df_ml.get("members", np.nan), errors="coerce")

threshold = df_ml["members"].quantile(HIT_Q)
df_ml["is_hit"] = (df_ml["members"] >= threshold).astype(int)

print("Hit threshold (members) =", threshold.round(2))
print("Positive rate =", df_ml["is_hit"].mean().round(4))


In [ ]:
# Show examples: borderline hits/non-hits + top hits overall

cols_to_show = [c for c in [
    "title", "type", "season", "year", "episodes",
    "genres", "themes", "demographics",
    "members", "is_hit"
] if c in df_ml.columns]

print(f"Hit cutoff (members) = {threshold:,.1f} | HIT_Q = {HIT_Q} (top {(1-HIT_Q)*100:.0f}%)")

# 5 hits closest to the cutoff (just above)
hits_near_cutoff = (
    df_ml[df_ml["is_hit"] == 1]
    .assign(dist_above=lambda x: x["members"] - threshold)
    .sort_values("dist_above", ascending=True)
    .head(5)
)

# 5 non-hits closest to the cutoff (just below)
nonhits_near_cutoff = (
    df_ml[df_ml["is_hit"] == 0]
    .assign(dist_below=lambda x: threshold - x["members"])
    .sort_values("dist_below", ascending=True)
    .head(5)
)

# 5 top hits overall (largest members)
top_hits_overall = (
    df_ml[df_ml["is_hit"] == 1]
    .sort_values("members", ascending=False)
    .head(5)
)

print("\n--- 5 HIT examples (just above the cutoff) ---")
display(hits_near_cutoff[cols_to_show])

print("\n--- 5 NON-HIT examples (just below the cutoff) ---")
display(nonhits_near_cutoff[cols_to_show])

print("\n--- 5 TOP HIT examples (highest members overall) ---")
display(top_hits_overall[cols_to_show])



In [ ]:
from sklearn.model_selection import train_test_split

BASE_CATS = ["type", "season", "source", "rating", "status"]
BASE_NUMS = ["year", "episodes"]
MULTI_COLS = ["genres", "themes", "demographics"]

BASE_CATS = [c for c in BASE_CATS if c in df_ml.columns]
BASE_NUMS = [c for c in BASE_NUMS if c in df_ml.columns]
MULTI_COLS = [c for c in MULTI_COLS if c in df_ml.columns]

FEATURE_COLS = BASE_CATS + BASE_NUMS + MULTI_COLS
TARGET_COL = "is_hit"

df_feat = df_ml[FEATURE_COLS + [TARGET_COL]].copy()

train_df, valid_df = train_test_split(
    df_feat,
    test_size=0.2,
    random_state=42,
    stratify=df_feat[TARGET_COL]
)

print("Train shape:", train_df.shape, "Valid shape:", valid_df.shape)


### Top‑K multi-hot encoding

We build the Top‑K vocab on **training only** to avoid leakage and feature mismatch.

In [ ]:
def build_topk_vocab(series, k=30):
    exploded = series.apply(parse_multilabel).explode()
    exploded = exploded.dropna()
    exploded = exploded[exploded.astype(str).str.len() > 0]
    return exploded.value_counts().head(k).index.tolist()

def add_multi_hot(df_in, col, vocab):
    df_out = df_in.copy()
    token_sets = df_out[col].apply(parse_multilabel).apply(set)
    for t in vocab:
        df_out[f"{col}__{t}"] = token_sets.apply(lambda s: 1 if t in s else 0)
    return df_out

TOPK = 30

train_X = train_df.drop(columns=[TARGET_COL]).copy()
valid_X = valid_df.drop(columns=[TARGET_COL]).copy()
y_train = train_df[TARGET_COL].copy()
y_valid = valid_df[TARGET_COL].copy()

vocab_map = {}
for col in MULTI_COLS:
    vocab = build_topk_vocab(train_X[col], k=TOPK)
    vocab_map[col] = vocab
    train_X = add_multi_hot(train_X, col, vocab)
    valid_X = add_multi_hot(valid_X, col, vocab)

# Drop original multi-label columns
train_X = train_X.drop(columns=MULTI_COLS, errors="ignore")
valid_X = valid_X.drop(columns=MULTI_COLS, errors="ignore")

X_train, X_valid = train_X, valid_X

print("X_train shape:", X_train.shape, "X_valid shape:", X_valid.shape)


---
## 6) Modeling: Experimenting with two missing value strategies

We will evaluate two approaches:
- **Impute in pipeline** (recommended for deployment)
- **Drop rows with missing values** (experiment-friendly)


In [ ]:
# Helper: drop rows with NaNs in the used feature columns (only when strategy='drop')
def apply_missing_strategy(X_train, y_train, X_valid, y_valid, cols_check, strategy):
    if strategy != "drop":
        return X_train, y_train, X_valid, y_valid, 0, 0

    tr_mask = X_train[cols_check].isna().any(axis=1)
    va_mask = X_valid[cols_check].isna().any(axis=1)

    dropped_train = int(tr_mask.sum())
    dropped_valid = int(va_mask.sum())

    X_train2 = X_train.loc[~tr_mask].copy()
    y_train2 = y_train.loc[~tr_mask].copy()
    X_valid2 = X_valid.loc[~va_mask].copy()
    y_valid2 = y_valid.loc[~va_mask].copy()

    return X_train2, y_train2, X_valid2, y_valid2, dropped_train, dropped_valid


In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses = []
    y_true = []
    y_prob = []

    for batch in loader:
        # Expected batch format from your AnimeDataset:
        # batch = (cats, ml_feats, nums, y)   OR   (cats, nums, ml_feats, y)
        # We'll handle both by looking at types
        if isinstance(batch, (list, tuple)) and len(batch) == 4:
            a, b, c, y = batch
            # Heuristic: ml_feats is usually a dict
            if isinstance(b, dict):
                cats, ml_feats, nums = a, b, c
            elif isinstance(c, dict):
                cats, nums, ml_feats = a, b, c
            else:
                raise ValueError("Can't detect ml_feats dict in batch.")
        else:
            raise ValueError("Unexpected batch structure from DataLoader.")

        cats = cats.to(device)
        nums = nums.to(device)
        y = y.to(device).float().view(-1, 1)

        ml_feats = {k: v.to(device) for k, v in ml_feats.items()}

        logits = model(cats, ml_feats, nums)
        loss = criterion(logits, y)

        prob = torch.sigmoid(logits)

        losses.append(loss.item())
        y_true.append(y.detach().cpu().numpy())
        y_prob.append(prob.detach().cpu().numpy())

    y_true = np.vstack(y_true).ravel()
    y_prob = np.vstack(y_prob).ravel()

    # Guard: if valid has only one class in rare cases, roc_auc_score will error
    roc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    pr  = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan

    return float(np.mean(losses)), float(roc), float(pr)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

CAT_COLS = BASE_CATS
NUM_COLS = BASE_NUMS

final_cat = [c for c in CAT_COLS if c in X_train.columns]
final_num = [c for c in NUM_COLS if c in X_train.columns]
final_bin = [c for c in X_train.columns if "__" in c]

cols_check = final_cat + final_num + final_bin

def fit_eval(strategy):
    X_tr, y_tr, X_va, y_va, dtr, dva = apply_missing_strategy(
        X_train, y_train, X_valid, y_valid, cols_check, strategy
    )

    if strategy == "impute":
        # Impute in pipeline (deployment-friendly)
        cat_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ])
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler())
        ])
        bin_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0))
        ])
    else:
        # Drop rows first, then no imputers needed
        cat_pipe = Pipeline(steps=[("ohe", OneHotEncoder(handle_unknown="ignore"))])
        num_pipe = Pipeline(steps=[("scaler", StandardScaler())])
        bin_pipe = "passthrough"

    preprocess = ColumnTransformer(
        transformers=[
            ("cats", cat_pipe, final_cat),
            ("nums", num_pipe, final_num),
            ("bins", bin_pipe, final_bin),
        ],
        remainder="drop",
    )

    logreg = Pipeline(steps=[
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=2000))
    ])

    rf = Pipeline(steps=[
        ("prep", preprocess),
        ("clf", RandomForestClassifier(n_estimators=400, random_state=SEED, n_jobs=-1))
    ])

    logreg.fit(X_tr, y_tr)
    rf.fit(X_tr, y_tr)

    pred_lr = logreg.predict_proba(X_va)[:, 1]
    pred_rf = rf.predict_proba(X_va)[:, 1]

    return {
        "strategy": strategy,
        "dropped_train_rows": dtr,
        "dropped_valid_rows": dva,
        "train_rows_used": len(X_tr),
        "valid_rows_used": len(X_va),
        "roc_auc_logreg": roc_auc_score(y_va, pred_lr),
        "pr_auc_logreg": average_precision_score(y_va, pred_lr),
        "roc_auc_rf": roc_auc_score(y_va, pred_rf),
        "pr_auc_rf": average_precision_score(y_va, pred_rf),
    }


---
## 7) Comparison table + delta view (drop − impute)


In [ ]:
results = [fit_eval("impute"), fit_eval("drop")]
results_df = pd.DataFrame(results)
display(results_df)

# delta view: (drop - impute)
deltas = results_df.set_index("strategy").loc["drop"] - results_df.set_index("strategy").loc["impute"]
display(deltas.to_frame("drop_minus_impute"))


> Note: The output of the AutoPick is above but for the purposes of deployment will go with impute. The difference between the strategies is small.

> Note: Now that we compared strategies above, we also **re-fit** a final baseline model below. For deployment, we prefer **impute** so the API can handle missing inputs.


---
## 7b) Re-fit models using the selected missing-value strategy (Impute vs Drop)

Above, we **compared** both strategies. In this section, we **re-train** (re-model) the pipelines using the strategy you choose.

- **Strategy = `impute`** is recommended for deployment (your service can accept incomplete inputs).
- **Strategy = `drop`** is only for analysis/experiments (you can't "drop the row" at inference time).

You can:
1) manually choose `SELECTED_STRATEGY` and `SELECTED_MODEL`, or  
2) let the notebook auto-pick the best strategy by validation ROC-AUC.


In [ ]:
# --- Choose strategy/model ---
AUTO_PICK = False                # set False to use manual selections below
SELECTED_STRATEGY = "impute"     # "impute" or "drop"  (used when AUTO_PICK=False)
SELECTED_MODEL = "rf"            # "rf" or "logreg"    (used when AUTO_PICK=False)

# --- Auto pick logic (based on validation ROC-AUC) ---
# results_df exists from the comparison section (cell above)
if AUTO_PICK:
    # pick best strategy for RF by ROC-AUC, tie-breaker PR-AUC
    best_rf = results_df.sort_values(["roc_auc_rf", "pr_auc_rf"], ascending=False).iloc[0]
    best_lr = results_df.sort_values(["roc_auc_logreg", "pr_auc_logreg"], ascending=False).iloc[0]

    # pick best overall between RF and LogReg by ROC-AUC (tie-break PR-AUC)
    if (best_rf["roc_auc_rf"], best_rf["pr_auc_rf"]) >= (best_lr["roc_auc_logreg"], best_lr["pr_auc_logreg"]):
        SELECTED_STRATEGY = best_rf["strategy"]
        SELECTED_MODEL = "rf"
    else:
        SELECTED_STRATEGY = best_lr["strategy"]
        SELECTED_MODEL = "logreg"

print("Selected strategy:", SELECTED_STRATEGY)
print("Selected model   :", SELECTED_MODEL)

# --- Build the preprocessing + model pipeline for the selected strategy ---
# fit_eval() already builds/uses the same logic; we create a reusable builder here
def build_pipeline(strategy: str, model_name: str):
    # strategy controls imputers; model_name chooses estimator
    if strategy == "impute":
        cat_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ])
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler())
        ])
        bin_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0))
        ])
    else:
        cat_pipe = Pipeline(steps=[("ohe", OneHotEncoder(handle_unknown="ignore"))])
        num_pipe = Pipeline(steps=[("scaler", StandardScaler())])
        bin_pipe = "passthrough"

    preprocess = ColumnTransformer(
        transformers=[
            ("cats", cat_pipe, final_cat),
            ("nums", num_pipe, final_num),
            ("bins", bin_pipe, final_bin),
        ],
        remainder="drop",
    )

    if model_name == "rf":
        est = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
    else:
        est = LogisticRegression(max_iter=2000)

    return Pipeline(steps=[("prep", preprocess), ("clf", est)])

# Apply drop strategy (if needed) before fitting/evaluating
X_tr, y_tr, X_va, y_va, dtr, dva = apply_missing_strategy(
    X_train, y_train, X_valid, y_valid, cols_check, SELECTED_STRATEGY
)

final_pipe = build_pipeline(SELECTED_STRATEGY, SELECTED_MODEL)
final_pipe.fit(X_tr, y_tr)

# Evaluate on the validation set (same split as above)
pred = final_pipe.predict_proba(X_va)[:, 1]
val_roc = roc_auc_score(y_va, pred)
val_pr  = average_precision_score(y_va, pred)

print(f"[REFIT] strategy={SELECTED_STRATEGY} model={SELECTED_MODEL} "
      f"dropped_train={dtr} dropped_valid={dva} ROC-AUC={val_roc:.4f} PR-AUC={val_pr:.4f}")

# --- Save artifacts (Sklearn baseline) ---
# Save under repo-root artifacts/ (safe even if you run from notebooks/)
repo_root = p  # from the earlier path-resolution cell
ART_DIR_SK = repo_root / "artifacts_sklearn"
ART_DIR_SK.mkdir(parents=True, exist_ok=True)

model_path = ART_DIR_SK / f"hit_baseline_{SELECTED_MODEL}_{SELECTED_STRATEGY}.joblib"
joblib.dump(final_pipe, model_path)

meta = {
    "selected_strategy": SELECTED_STRATEGY,
    "selected_model": SELECTED_MODEL,
    "val_roc_auc": float(val_roc),
    "val_pr_auc": float(val_pr),
    "dropped_train_rows": int(dtr),
    "dropped_valid_rows": int(dva),
    "cat_cols": final_cat,
    "num_cols": final_num,
    "bin_cols": final_bin,
    "topk": int(TOPK) if "TOPK" in globals() else None,
    "vocab_map": vocab_map,  # for reproducibility / inspection
}

meta_path = ART_DIR_SK / f"hit_baseline_{SELECTED_MODEL}_{SELECTED_STRATEGY}_meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved model:", model_path)
print("Saved meta :", meta_path)



---
## 8) Deep Learning + ONNX + Deployment (Serverless + Kubernetes)

Before running the deep learning section, we will do a small **compatibility prep**:
- ensure multi-label columns are lists (not raw strings)
- provide variable names expected by the DL/ONNX/deployment code


In [ ]:
# Compatibility prep for Deep Learning section

import math
import random
import json
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from sklearn.metrics import roc_auc_score, average_precision_score

# If you ran the notebook top-to-bottom, these should exist already.
# We alias them to the names used later in the end-to-end sections.

# Columns
CAT_COLS = BASE_CATS
NUM_COLS = BASE_NUMS

# The DL section expects a dict called `vocabs` for MULTI_COLS
# where vocabs[col] is a list of Top-K tokens.
vocabs = vocab_map

# Make sure multi-label columns are lists (not raw strings) in train/valid frames
for c in MULTI_COLS:
    if c in train_df.columns:
        train_df[c] = train_df[c].apply(parse_multilabel)
    if c in valid_df.columns:
        valid_df[c] = valid_df[c].apply(parse_multilabel)

print("Ready for DL section.")
print("CAT_COLS:", CAT_COLS)
print("NUM_COLS:", NUM_COLS)
print("MULTI_COLS:", MULTI_COLS)


## 9) Deep learning model (PyTorch tabular)

We’ll do a **tabular neural net** that:
- embeds categorical features (type/season/source/rating/status)
- embeds multi-label tokens (genres/themes/demographics) and averages them
- concatenates numerics
- passes through an MLP with dropout

In [ ]:
# import torch
# import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader

In [ ]:
torch.manual_seed(SEED)

# If you ever use CUDA in the future:
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Apple Silicon (MPS) note: exact determinism is not always guaranteed,
# but seeding still improves repeatability.
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    # No extra deterministic flags like cuDNN, but keep the seed set
    pass

print("Torch SEED set to:", SEED)


In [ ]:
# Build vocabularies for embedding layers
def build_category_vocab(series: pd.Series, min_count: int = 1):
    counts = series.fillna("unknown").astype(str).value_counts()
    tokens = ["<PAD>", "<UNK>"] + [t for t, c in counts.items() if c >= min_count]
    stoi = {t:i for i,t in enumerate(tokens)}
    return tokens, stoi

cat_vocabs = {}
cat_stoi = {}
for c in CAT_COLS:
    if c in train_df.columns:
        tokens, stoi = build_category_vocab(train_df[c])
        cat_vocabs[c] = tokens
        cat_stoi[c] = stoi

# Multi-label vocab (top-K) -> index; add PAD/UNK
ml_vocabs = {}
ml_stoi = {}
for c in MULTI_COLS:
    if c in train_df.columns:
        vocab = ["<PAD>", "<UNK>"] + vocabs[c]
        stoi = {t:i for i,t in enumerate(vocab)}
        ml_vocabs[c] = vocab
        ml_stoi[c] = stoi

# Numeric normalization stats (from train only)
num_means = train_df[NUM_COLS].mean(numeric_only=True)
num_stds  = train_df[NUM_COLS].std(numeric_only=True).replace(0, 1.0)

In [ ]:
def encode_cat(value: Optional[str], stoi: Dict[str,int]) -> int:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        value = "unknown"
    s = str(value)
    return stoi.get(s, stoi["<UNK>"])

def encode_multilabel(items: List[str], stoi: Dict[str,int], max_len: int = 10) -> List[int]:
    # take up to max_len tokens, pad
    ids = [stoi.get(t, stoi["<UNK>"]) for t in items[:max_len]]
    if len(ids) < max_len:
        ids = ids + [stoi["<PAD>"]] * (max_len - len(ids))
    return ids

class AnimeDataset(Dataset):
    def __init__(self, df_in: pd.DataFrame, max_ml_len: int = 10):
        self.df = df_in.reset_index(drop=True)
        self.max_ml_len = max_ml_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.loc[idx]

        cat_feats = []
        for c in CAT_COLS:
            if c in self.df.columns:
                cat_feats.append(encode_cat(row.get(c), cat_stoi[c]))
        cat_feats = torch.tensor(cat_feats, dtype=torch.long)

        ml_feats = {}
        for c in MULTI_COLS:
            if c in self.df.columns:
                ml_feats[c] = torch.tensor(
                    encode_multilabel(row.get(c, []), ml_stoi[c], max_len=self.max_ml_len),
                    dtype=torch.long
                )

        nums = []
        for c in NUM_COLS:
            v = row.get(c)
            v = float(v) if pd.notna(v) else float(num_means[c])
            v = (v - float(num_means[c])) / float(num_stds[c])
            nums.append(v)
        nums = torch.tensor(nums, dtype=torch.float32)

        y = torch.tensor(float(row[TARGET_COL]), dtype=torch.float32)
        return cat_feats, ml_feats, nums, y

In [ ]:
class TabularHitNet(nn.Module):
    def __init__(self,
                 cat_cardinalities: List[int],
                 cat_emb_dim: int = 16,
                 ml_cardinalities: Dict[str,int] = None,
                 ml_emb_dim: int = 16,
                 num_dim: int = 2,
                 hidden_dims: List[int] = [128, 64],
                 dropout: float = 0.2):
        super().__init__()

        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(card, cat_emb_dim) for card in cat_cardinalities
        ])

        self.ml_embeddings = nn.ModuleDict({
            k: nn.Embedding(v, ml_emb_dim, padding_idx=0) for k, v in (ml_cardinalities or {}).items()
        })

        in_dim = len(cat_cardinalities) * cat_emb_dim + len(self.ml_embeddings) * ml_emb_dim + num_dim

        layers = []
        d = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers += [nn.Linear(d, 1)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, cats: torch.Tensor, ml_feats: Dict[str, torch.Tensor], nums: torch.Tensor):
        # cats: [B, n_cat]
        emb_cat = []
        for i, emb in enumerate(self.cat_embeddings):
            emb_cat.append(emb(cats[:, i]))
        emb_cat = torch.cat(emb_cat, dim=1) if emb_cat else torch.empty((cats.size(0), 0), device=cats.device)

        emb_ml = []
        for k, emb in self.ml_embeddings.items():
            x = ml_feats[k]  # [B, L]
            e = emb(x)       # [B, L, D]
            # average over non-pad tokens
            mask = (x != 0).float().unsqueeze(-1)  # [B, L, 1]
            denom = mask.sum(dim=1).clamp(min=1.0)
            pooled = (e * mask).sum(dim=1) / denom  # [B, D]
            emb_ml.append(pooled)
        emb_ml = torch.cat(emb_ml, dim=1) if emb_ml else torch.empty((cats.size(0), 0), device=cats.device)

        x = torch.cat([emb_cat, emb_ml, nums], dim=1)
        logits = self.mlp(x).squeeze(1)
        return logits

In [ ]:
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
from sklearn.model_selection import train_test_split

# df_feat contains features + TARGET_COL already
# train_df + valid_df already exist in your notebook, but we'll rebuild a clean 3-way split here.

train_df, temp_df = train_test_split(
    df_feat,
    test_size=0.2,            # 80% train, 20% temp
    random_state=SEED,
    stratify=df_feat[TARGET_COL]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,            # split temp into 10% valid, 10% test
    random_state=SEED,
    stratify=temp_df[TARGET_COL]
)

print("Train:", train_df.shape, "Valid:", valid_df.shape, "Test:", test_df.shape)
print("Hit rate (train/valid/test):",
      train_df[TARGET_COL].mean().round(4),
      valid_df[TARGET_COL].mean().round(4),
      test_df[TARGET_COL].mean().round(4))


In [ ]:
# DataLoaders
BATCH_SIZE = 256
MAX_ML_LEN = 10

train_ds = AnimeDataset(train_df, max_ml_len=MAX_ML_LEN)
valid_ds = AnimeDataset(valid_df, max_ml_len=MAX_ML_LEN)
test_ds  = AnimeDataset(test_df,  max_ml_len=MAX_ML_LEN)



def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, # keep 0 for maximum reproducibility; increase later if needed
    worker_init_fn=seed_worker, 
    generator=g,
)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,  num_workers=0,
    worker_init_fn=seed_worker,
    generator=g,
)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,  num_workers=0,
    worker_init_fn=seed_worker,
    generator=g,
)

cat_cards = [len(cat_vocabs[c]) for c in CAT_COLS if c in cat_vocabs]
ml_cards  = {c: len(ml_vocabs[c]) for c in MULTI_COLS if c in ml_vocabs}

model = TabularHitNet(
    cat_cardinalities=cat_cards,
    cat_emb_dim=16,
    ml_cardinalities=ml_cards,
    ml_emb_dim=16,
    num_dim=len(NUM_COLS),
    hidden_dims=[128, 64],
    dropout=0.3,   # tune this
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # tune lr

In [ ]:
history = {
    "epoch": [],
    "train_loss": [],
    "valid_loss": [],
    "valid_roc_auc": [],
    "valid_pr_auc": [],
}


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

@torch.no_grad()
def eval_loader(model, loader, criterion=None):
    model.eval()
    ys, ps = [], []
    losses = []

    for cats, ml_feats, nums, y in loader:
        cats = cats.to(device)
        nums = nums.to(device)
        ml_feats = {k: v.to(device) for k, v in ml_feats.items()}
        y = y.to(device)

        logits = model(cats, ml_feats, nums)
        prob = torch.sigmoid(logits).detach().cpu().numpy()

        ys.append(y.detach().cpu().numpy())
        ps.append(prob)

        if criterion is not None:
            losses.append(criterion(logits, y).item())

    y_true = np.concatenate(ys).ravel()
    y_prob = np.concatenate(ps).ravel()

    out = {
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        "pr_auc": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
    }
    if criterion is not None:
        out["loss"] = float(np.mean(losses)) if losses else np.nan

    return out


def train_epochs(model, train_loader, valid_loader, epochs=15):
    best = (-1, None)

    history = {
        "epoch": [],
        "train_loss": [],
        "valid_loss": [],
        "valid_roc_auc": [],
        "valid_pr_auc": [],
    }

    for epoch in range(1, epochs+1):
        model.train()
        losses = []

        for cats, ml_feats, nums, y in train_loader:
            cats = cats.to(device)
            nums = nums.to(device)
            ml_feats = {k: v.to(device) for k, v in ml_feats.items()}
            y = y.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(cats, ml_feats, nums)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        train_loss = float(np.mean(losses))
        metrics = eval_loader(model, valid_loader, criterion=criterion)

        # --- record for plots ---
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["valid_loss"].append(metrics.get("loss", np.nan))
        history["valid_roc_auc"].append(metrics["roc_auc"])
        history["valid_pr_auc"].append(metrics["pr_auc"])

        print(
            f"epoch={epoch:02d} "
            f"train_loss={train_loss:.4f} "
            f"valid_loss={history['valid_loss'][-1]:.4f} "
            f"valid_roc_auc={metrics['roc_auc']:.4f} "
            f"valid_pr_auc={metrics['pr_auc']:.4f}"
        )

        if metrics["roc_auc"] > best[0]:
            best = (metrics["roc_auc"], {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})

    return best, history


# best_auc, best_state = train_epochs(model, train_loader, valid_loader, epochs=15)
# best_auc

(best_auc, best_state), history = train_epochs(model, train_loader, valid_loader, epochs=15)
best_auc


In [ ]:
# Load best weights and evaluate on test set
model.load_state_dict(best_state)
test_metrics = eval_loader(model, test_loader)
test_metrics

## 9a) Plotting the results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

hist_df = pd.DataFrame(history)
display(hist_df)

# Loss plot
plt.figure(figsize=(8,4))
plt.plot(hist_df["epoch"], hist_df["train_loss"], label="train_loss")
plt.plot(hist_df["epoch"], hist_df["valid_loss"], label="valid_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Loss vs Epoch")
plt.legend()
plt.tight_layout()
plt.show()

# ROC-AUC plot
plt.figure(figsize=(8,4))
plt.plot(hist_df["epoch"], hist_df["valid_roc_auc"], label="valid_roc_auc")
plt.xlabel("epoch")
plt.ylabel("ROC-AUC")
plt.title("Validation ROC-AUC vs Epoch")
plt.legend()
plt.tight_layout()
plt.show()

# PR-AUC plot
plt.figure(figsize=(8,4))
plt.plot(hist_df["epoch"], hist_df["valid_pr_auc"], label="valid_pr_auc")
plt.xlabel("epoch")
plt.ylabel("PR-AUC")
plt.title("Validation PR-AUC vs Epoch")
plt.legend()
plt.tight_layout()
plt.show()


## 10) Save artifacts (model + preprocessing config)

We’ll save:
- model weights (PyTorch) **and/or** ONNX model for inference
- vocabularies + normalization stats (so preprocessing is identical in training and serving)

We’ll prefer ONNX for serverless + small CPU inference.

In [ ]:
# --- Artifacts directory (absolute path for clarity) ---
# ART_DIR is defined above as REPO_ROOT / "artifacts"

ART_DIR.mkdir(parents=True, exist_ok=True)

# --- Save PyTorch weights ---
torch_fname = "hitnet.pt"
torch_path = ART_DIR / torch_fname
torch.save(model.state_dict(), torch_path)

# --- Save preprocessing config ---
preproc_fname = "preproc.json"
preproc_path = ART_DIR / preproc_fname

preproc = {
    "cat_cols": CAT_COLS,
    "num_cols": NUM_COLS,
    "multi_cols": MULTI_COLS,
    "cat_stoi": cat_stoi,
    "ml_stoi": ml_stoi,
    "num_means": num_means.to_dict(),
    "num_stds": num_stds.to_dict(),
    "max_ml_len": MAX_ML_LEN,
    "model_config": {
        "cat_cards": cat_cards,
        "cat_emb_dim": 16,
        "ml_cards": ml_cards,
        "ml_emb_dim": 16,
        "num_dim": len(NUM_COLS),
        "hidden_dims": [128, 64],
        "dropout": 0.3,
    }
}

with open(preproc_path, "w") as f:
    json.dump(preproc, f, indent=2)

# --- Confirmation prints ---
def _fmt_size(p: Path) -> str:
    try:
        return f"{p.stat().st_size:,} bytes"
    except FileNotFoundError:
        return "MISSING"

print("\n✅ Artifacts saved successfully!")
print(f"📁 Directory: {ART_DIR}")

print("\nFiles:")
print(f" - {torch_fname}  -> {torch_path}  ({_fmt_size(torch_path)})")
print(f" - {preproc_fname} -> {preproc_path} ({_fmt_size(preproc_path)})")

# Optional: list everything in artifacts folder
print("\n📦 Contents of artifacts folder:")
for p in sorted(ART_DIR.glob("*")):
    if p.is_file():
        print(f" - {p.name} ({_fmt_size(p)})")


## 11) Export to ONNX (recommended for serverless)

Why ONNX?
- Inference runtime is smaller & faster on CPU
- Works nicely in AWS Lambda container images with `onnxruntime`

We’ll export a model that accepts:
- categorical ids tensor
- numeric tensor
- multi-label id tensors (one per field)

(You can also export a version that takes one flattened float vector if you prefer.)

In [ ]:
# =========================
# ONNX export (Fix 1: legacy exporter via dynamo=False) — FULL WORKING SNIPPET
# =========================

import os
import json
import numpy as np
from pathlib import Path
from typing import List

import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

# --- Artifacts directory (use Path to avoid str/Path bugs) ---
# ART_DIR is defined above as REPO_ROOT / "artifacts"

ART_DIR.mkdir(parents=True, exist_ok=True)

def _fmt_size(p) -> str:
    p = Path(p)
    try:
        return f"{p.stat().st_size:,} bytes"
    except FileNotFoundError:
        return "MISSING"

# --- ONNX output path ---
onnx_path = ART_DIR / "hitnet.onnx"

# --- Wrap forward signature to fixed arg order (ONNX likes fixed args) ---
class ExportWrapper(nn.Module):
    def __init__(self, base, ml_keys: List[str]):
        super().__init__()
        self.base = base
        self.ml_keys = ml_keys

    def forward(self, cats, nums, *ml_tensors):
        ml_feats = {k: t for k, t in zip(self.ml_keys, ml_tensors)}
        # IMPORTANT: must match your model signature
        # Your TabularHitNet was called like: model(cats, ml_feats, nums)
        return self.base(cats, ml_feats, nums)

# --- Determine ML keys (must match training/preproc order) ---
ml_keys = list(ml_cards.keys())  # e.g. ["genres", "themes", "demographics"]

# --- Export on CPU for stability (Apple Silicon/MPS especially) ---
wrapped_cpu = ExportWrapper(model, ml_keys).cpu().eval()

# --- Dummy inputs for tracing ---
B = 2
dummy_cats = torch.zeros((B, len(cat_cards)), dtype=torch.long)          # int64
dummy_nums = torch.zeros((B, len(NUM_COLS)), dtype=torch.float32)        # float32
dummy_ml   = {k: torch.zeros((B, MAX_ML_LEN), dtype=torch.long) for k in ml_keys}

input_names  = ["cats", "nums"] + [f"ml_{k}" for k in ml_keys]
output_names = ["logits"]

dynamic_axes = {name: {0: "batch"} for name in input_names + output_names}

# --- Export ---
torch.onnx.export(
    wrapped_cpu,
    (dummy_cats, dummy_nums, *[dummy_ml[k] for k in ml_keys]),
    str(onnx_path),                         # torch.onnx.export prefers str
    input_names=input_names,
    output_names=output_names,
    dynamic_axes=dynamic_axes,
    opset_version=17,
    dynamo=False,                           # ✅ avoids dynamic_shapes conversion error
)

# --- Validate ONNX ---
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

# --- Quick ONNXRuntime load test + print IO ---
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print("✅ ONNX exported successfully!")
print(f"📁 Artifacts directory: {ART_DIR}")
print(f" - hitnet.onnx -> {onnx_path} ({_fmt_size(onnx_path)})")
print("ONNX inputs :", [i.name for i in sess.get_inputs()])
print("ONNX outputs:", [o.name for o in sess.get_outputs()])

# Optional: list artifacts folder contents
print("\n📦 Contents of artifacts folder:")
for p in sorted(ART_DIR.glob("*")):
    if p.is_file():
        print(f" - {p.name} ({_fmt_size(p)})")


## 12) Inference function (shared by API + Lambda)

We’ll implement preprocessing in **pure Python + numpy** and run ONNX inference.
That keeps deployment images smaller than shipping full PyTorch.

In [ ]:
import numpy as np

def preprocess_record(record: Dict, preproc: Dict) -> Tuple[np.ndarray, np.ndarray, Dict[str, np.ndarray]]:
    # categorical ids
    cats = []
    for c in preproc["cat_cols"]:
        val = record.get(c, "unknown")
        cats.append(preproc["cat_stoi"][c].get(str(val), preproc["cat_stoi"][c]["<UNK>"]))
    cats = np.array(cats, dtype=np.int64)[None, :]  # [1, n_cat]

    # numeric normalized
    nums = []
    for c in preproc["num_cols"]:
        v = record.get(c, preproc["num_means"][c])
        try:
            v = float(v)
        except Exception:
            v = float(preproc["num_means"][c])
        v = (v - float(preproc["num_means"][c])) / float(preproc["num_stds"][c])
        nums.append(v)
    nums = np.array(nums, dtype=np.float32)[None, :]  # [1, n_num]

    # multi-label padded ids
    ml = {}
    L = int(preproc["max_ml_len"])
    for c in preproc["multi_cols"]:
        items = record.get(c, [])
        if items is None:
            items = []
        if isinstance(items, str):
            items = parse_multilabel(items)
        stoi = preproc["ml_stoi"].get(c, None)
        if stoi is None:
            continue
        ids = [stoi.get(t, stoi["<UNK>"]) for t in items[:L]]
        if len(ids) < L:
            ids += [stoi["<PAD>"]] * (L - len(ids))
        ml[c] = np.array(ids, dtype=np.int64)[None, :]  # [1, L]
    return cats, nums, ml

def predict_proba_onnx(record: Dict, onnx_path: str, preproc_path: str) -> float:
    with open(preproc_path, "r") as f:
        preproc = json.load(f)
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

    cats, nums, ml = preprocess_record(record, preproc)
    feed = {"cats": cats, "nums": nums}
    for k, v in ml.items():
        feed[f"ml_{k}"] = v

    logits = sess.run(["logits"], feed)[0]
    logit = np.asarray(logits).reshape(-1)[0]          # ✅ shape-safe
    score = 1.0 / (1.0 + np.exp(-logit))               # sigmoid
    return float(score)



example = {
    "type": "TV",
    "season": "Spring",
    "year": 2025,
    "episodes": 12,
    "source": "Manga",
    "rating": "PG-13",
    "status": "Upcoming",
    "genres": ["Action", "Adventure"],
    "themes": ["School"],
    "demographics": ["Shounen"]
}

predict_proba_onnx(example, onnx_path, os.path.join(ART_DIR, "preproc.json"))

---
# Scripts

# The sections below are for creating base scripts for the repo. However this step has already been done and these files live in the respective rolders within the repo along with additional config / processing scripts

### Turn this notebook into scripts

Minimum scripts (copy/paste the relevant cells):
- `train.py`
  - load data, create label
  - train best ML model + deep learning model
  - choose best
  - export artifacts (`preproc.json`, `hitnet.onnx`)
- `predict.py`
  - reads JSON from CLI, prints probability
- `serve.py`
  - FastAPI app as shown
- `lambda_function.py`
  - Lambda handler as shown
- `k8s/*.yaml`

---

## 13) Web service (FastAPI)

Endpoints:
- `GET /health` → 200 OK
- `POST /predict` → JSON → probability + label

You will copy this into `serve.py`.

```bash
# serve.py (copy this cell into a script)

from __future__ import annotations

from typing import Dict, List, Optional

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

try:
    from .predictor import predict as predict_any  # type: ignore
except Exception:  # pragma: no cover
    from src.predictor import predict as predict_any  # type: ignore


class AnimeRequest(BaseModel):
    type: Optional[str] = None
    season: Optional[str] = None
    year: Optional[int] = None
    episodes: Optional[int] = None
    source: Optional[str] = None
    rating: Optional[str] = None
    status: Optional[str] = None

    genres: List[str] = Field(default_factory=list)
    themes: List[str] = Field(default_factory=list)
    demographics: List[str] = Field(default_factory=list)
    studios: List[str] = Field(default_factory=list)


class PredictionResponse(BaseModel):
    hit_probability: float
    hit: bool
    backend: Optional[str] = None
    threshold: Optional[float] = None


app = FastAPI(title="Anime Hit Prediction API", version="1.0")


@app.get("/health")
def health() -> Dict[str, str]:
    return {"status": "ok"}


@app.post("/predict", response_model=PredictionResponse)
def predict(req: AnimeRequest) -> PredictionResponse:
    try:
        payload = req.model_dump()
    except Exception:
        payload = req.dict()

    try:
        out = predict_any(payload)
        return PredictionResponse(**out)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

```


### 13.1 Test locally

If you prefer one-liners, you can use the Makefile to do local testing as well:

> Note: If you have virtual environment running already I suggest removing the current virtual environment first before doing the local test. Open a new terminal and the command below:

```bash
rm -rf .venv
make install # Create virtual env
```

#### 1. Train & Export (or download artifacts) - Open New Terminal

```bash
make train
make export
```

#### 2. Server
```bash
make serve # uses PORT=9696 by default
# or: make serve PORT=8000
```
#### 3. Test Prediction Request - Open New Terminal
```bash
make health PORT=9696
make test PORT=9696

```

## 14) Docker (local deploy)

We build an image that serves FastAPI + ONNX Runtime.

#### Copy into Dockerfile or create dockerfile with code below:

```bash
FROM python:3.11-slim

# Keeps logs unbuffered and avoids .pyc files
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

# Default to the winning baseline model
ENV MODEL_BACKEND=rf

WORKDIR /app

# Install deps first (better Docker layer caching)
COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir --upgrade pip \
 && pip install --no-cache-dir -r requirements.txt

# Copy the rest of the repo
COPY . /app

# Default API port (matches README / Makefile)
EXPOSE 9696

# Start FastAPI via uvicorn.
# Note: we reference src.serve:app because the capstone layout uses src/serve.py
CMD ["uvicorn", "src.serve:app", "--host", "0.0.0.0", "--port", "9696"]

```

#### Build/run:

```bash
docker build -t anime-hit-api:latest .
docker run --rm -p 9696:9696 anime-hit-api:latest
```

Or Use the Makefile shortcut
```bash
make docker-build
make docker-run
```
> Note: On Apple Silicon, if you run into architecture mismatches:
```bash
docker buildx build --platform linux/amd64 -t anime-hit-api .
```

### Test

```bash
curl -X POST "http://localhost:9696/predict" \
  -H "Content-Type: application/json" \
  -d '{
    "type": "TV",
    "season": "Spring",
    "year": 2025,
    "episodes": 12,
    "source": "Manga",
    "rating": "PG-13",
    "status": "Upcoming",
    "genres": ["Action"],
    "themes": [],
    "demographics": []
  }'

## 15) Serverless deployment (AWS Lambda container) with ONNX Runtime

Pattern:
- package `lambda_function.py` with a `handler(event, context)`
- ship ONNX model + `preproc.json` inside the container
- expose via API Gateway (HTTP API)

This is intentionally similar to the Serverless Deep Learning module notebook you attached.

```bash
# lambda_function.py
# ------------------
import json

from .predictor import predict


def handler(event, context=None):
    """
    Supports two invocation styles:

    1) Direct payload:
       {"type":"TV", ...}

    2) Body-wrapped payload (API Gateway-ish):
       {"body":"{...json...}"} or {"body":{...}}
    """
    payload = event
    if isinstance(event, dict) and "body" in event:
        body = event["body"]
        payload = json.loads(body) if isinstance(body, str) else body

    try:
        out = predict(payload)
        return {
            "statusCode": 200,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps(out),
        }
    except Exception as e:
        return {
            "statusCode": 500,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({"error": str(e)}),
        }

```


#### Dockerfile (Lambda) — save as docker/Dockerfile.lambda
#### ------------------------------------------------
```bash
# AWS Lambda container image (serverless deep learning)
# Local test:
#   curl -X POST http://localhost:9000/2015-03-31/functions/function/invocations -d '{...}'
# Option B: Python 3.12 Lambda base image so we can install newer onnxruntime wheels.
FROM public.ecr.aws/lambda/python:3.12

# Install minimal inference deps
COPY requirements-lambda.txt .
RUN pip install --no-cache-dir -r requirements-lambda.txt

# Default backend for Lambda is RF; users can override MODEL_BACKEND=hitnet
ENV MODEL_BACKEND=rf

# Copy application code and (generated) artifacts
COPY src ./src
COPY artifacts ./artifacts

# Handler
CMD ["src.lambda_function.handler"]

```

Build and run locally with the Lambda Runtime Interface Emulator:

```bash
docker buildx build --platform linux/amd64 -f Dockerfile.lambda -t anime-hit-lambda .
docker run -p 9000:8080 anime-hit-lambda
```
<br>

> Alternative: Using Makefile shortcut
```bash
make docker-build-lambda
make docker-run-lambda
```

Test Invoke:
```bash
curl -X POST "http://localhost:9000/2015-03-31/functions/function/invocations" \
  -H "Content-Type: application/json" \
  -d '{
    "type": "TV",
    "season": "Spring",
    "year": 2025,
    "episodes": 12,
    "source": "Manga",
    "rating": "PG-13",
    "status": "Upcoming",
    "genres": ["Action"],
    "themes": ["School"],
    "demographics": ["Shounen"]
  }'
```

Deploy steps (AWS CLI outline):
1. Push image to ECR
2. Create Lambda from container image
3. Create HTTP API Gateway and integrate

## 16) Kubernetes deployment (kind/minikube) - Pick up here tomorrow 12/15/25

We’ll deploy the **same API container** to a cluster.

Artifacts to include:
- `k8s/deployment.yaml`
- `k8s/service.yaml`
- (optional) `k8s/hpa.yaml`

Your `deployment.yaml` references the Docker image you built/pushed.

In [ ]:
# k8s/deployment.yaml
# -------------------
# apiVersion: apps/v1
# kind: Deployment
# metadata:
#   name: anime-hit-api
# spec:
#   replicas: 1
#   selector:
#     matchLabels:
#       app: anime-hit-api
#   template:
#     metadata:
#       labels:
#         app: anime-hit-api
#     spec:
#       containers:
#         - name: api
#           image: anime-hit-api:latest
#           imagePullPolicy: IfNotPresent
#           ports:
#             - containerPort: 9696
#           env:
#             - name: MODEL_BACKEND
#               value: "rf"
#             - name: RF_MODEL_PATH
#               value: /app/artifacts/rf_pipeline.joblib
#             - name: RF_META_PATH
#               value: /app/artifacts/rf_meta.json
#             - name: RF_THRESHOLD_PATH
#               value: /app/artifacts/rf_threshold.json
#             # Optional HitNet backend (switch MODEL_BACKEND=hitnet)
#             - name: ONNX_PATH
#               value: /app/artifacts/hitnet.onnx
#             - name: PREPROC_PATH
#               value: /app/artifacts/preproc.json
#             - name: HITNET_THRESHOLD_PATH
#               value: /app/artifacts/hitnet_threshold.json
#             - name: THRESHOLD
#               value: "0.5"
# ---
# apiVersion: v1
# kind: Service
# metadata:
#   name: anime-hit-api-svc
# spec:
#   type: NodePort
#   selector:
#     app: anime-hit-api
#   ports:
#     - port: 9696
#       targetPort: 9696
#       nodePort: 30080


In [ ]:
# k8s/service.yaml
# ----------------
# apiVersion: v1
# kind: Service
# metadata:
#   name: anime-hit-api
# spec:
#   type: ClusterIP
#   selector:
#     app: anime-hit-api
#   ports:
#   - port: 80
#     targetPort: 9696

In [ ]:
# (optional) k8s/hpa.yaml
# -----------------------
# apiVersion: autoscaling/v2
# kind: HorizontalPodAutoscaler
# metadata:
#   name: anime-hit-api-hpa
# spec:
#   scaleTargetRef:
#     apiVersion: apps/v1
#     kind: Deployment
#     name: anime-hit-api
#   minReplicas: 2
#   maxReplicas: 6
#   metrics:
#   - type: Resource
#     resource:
#       name: cpu
#       target:
#         type: Utilization
#         averageUtilization: 70

### 16.1 Run on kind (local cluster)

```bash
kind create cluster
# Build & Load your local image into kind:
docker build -t anime-hit-api:latest .
kind load docker-image anime-hit-api:latest

# Deploy manifests
kubectl apply -f k8s/deployment.yaml
kubectl apply -f k8s/service.yaml

kubectl get pods
kubectl get svc
```

Test with port-forward:

```bash
kubectl port-forward svc/anime-hit-api 9696:80
curl -X POST "http://localhost:9696/predict" \
  -H "Content-Type: application/json" \
  -d '{
  "type":"TV",
  "season":"Spring",
  "year":2025,
  "episodes":12,
  "source":"Manga",
  "rating":"PG-13",
  "status":"Upcoming",
  "genres":["Action","Adventure"],
  "themes":["School"],
  "demographics":["Shounen"],
  "studios": []
}'
```

## 17) Keras alternative (if you prefer TF/Keras)

This notebook uses Keras/TensorFlow patterns (learning rate, checkpointing, dropout, etc.).
If you want an easy DL baseline in TF:

- Use `tf.keras.layers.StringLookup` + `Embedding` for categories
- Use `TextVectorization` for multi-label (or make multi-hot vectors)
- Use an MLP head

You can still export to ONNX (via `tf2onnx`) and reuse the same inference stack.